# 12 · 모델 비교 실험실 — 같은 데이터, 여러 모델
«AI 모델 도감»에 나온 모델들을 **같은 데이터 · 같은 나누기 · 같은 지표**로 한꺼번에 돌려 봅니다.
Ⅱ-04 «모델 고르는 다섯 걸음»의 4~5걸음을 코드로 해 보는 노트북입니다.

맨 먼저 **파일 → 드라이브에 사본 저장**을 하세요(내 드라이브에 저장돼야 고친 내용이 남습니다).
`____` 가 보이면 거기가 문제입니다. 빈칸 **바로 위 💡 힌트**를 읽고 채운 뒤 셀을 실행하세요(Shift + Enter).
교과서는 `files.upload()` 로 파일을 올리지만, 이 노트북은 **선생님 저장소에서 바로 읽어** 올릴 파일이 없습니다.

데이터: 남극 **펭귄 세 종**(Adelie · Chinstrap · Gentoo)의 부리 길이 · 부리 깊이 · 날개 길이 · 몸무게 — seaborn에 들어 있습니다.

## ① 불러오기 · 전처리

In [ ]:
# (그래프에 한글이 네모로 나오면 한 번만 실행)
!pip -q install koreanize-matplotlib
import koreanize_matplotlib

In [ ]:
import pandas as pd, seaborn as sns, matplotlib.pyplot as plt
df = sns.load_dataset('penguins').dropna()
X = df[['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']]
y = df['species']
print(X.shape, y.value_counts().to_dict())

## ② 분류 모델 여덟 개를 한 번에
💡 거리를 재는 모델(kNN · SVM · 신경망)은 단위를 맞춰야 공정합니다 → `StandardScaler` 를 앞에 붙이는 `make_pipeline`.
💡 교차 검증은 데이터를 다섯 번 나눠 다섯 번 시험 → `cv=5`.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier

S = StandardScaler
모델 = {
  '로지스틱 회귀':   make_pipeline(S(), LogisticRegression(max_iter=1000)),
  'k-최근접 이웃':   make_pipeline(S(), KNeighborsClassifier(5)),
  '의사결정트리':    DecisionTreeClassifier(max_depth=4, random_state=0),
  '랜덤 포레스트':   RandomForestClassifier(200, random_state=0),
  '그레이디언트 부스팅': GradientBoostingClassifier(random_state=0),
  'SVM':            make_pipeline(S(), SVC()),
  '나이브 베이즈':   GaussianNB(),
  '신경망(MLP)':     make_pipeline(S(), MLPClassifier((32, 16), max_iter=2000, random_state=0)),
}
결과 = {}
for 이름, m in 모델.items():
    점수 = cross_val_score(m, X, y, cv=____)
    결과[이름] = 점수.mean()
    print(f'{이름:12s} 평균 정확도 {점수.mean():.3f}  (±{점수.std():.3f})')

In [ ]:
pd.Series(결과).sort_values().plot.barh(xlim=(0.8, 1.0), title='cross-validation accuracy')
plt.show()

**생각해 보기 ①** 점수 차가 0.02 안쪽인 모델이 몇 개인가요? 그중 무엇을 고르겠나요? (설명하기 쉬운 쪽?)

## ③ 같은 데이터, 회귀 모델 — 몸무게 맞히기
이 표는 **종별로 줄지어 정렬**돼 있습니다. 섞지 않고 다섯 토막을 내면 «Gentoo만 모인 토막»으로 시험 보게 되어 점수가 엉망이 됩니다. 그래서 `KFold(shuffle=True)` 로 **섞어서** 나눕니다(`train_test_split` 은 원래 섞어 줍니다).

💡 회귀 점수는 R². `scoring='r2'`.

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import KFold
섞어나누기 = KFold(5, shuffle=True, random_state=0)
Xr = df[['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm']]
yr = df['body_mass_g']
회귀 = {'선형 회귀': LinearRegression(), '릿지': Ridge(1.0), 'kNN 회귀': make_pipeline(S(), KNeighborsRegressor(5)),
        '회귀 트리': DecisionTreeRegressor(max_depth=4, random_state=0), '랜덤 포레스트 회귀': RandomForestRegressor(200, random_state=0)}
for 이름, m in 회귀.items():
    점수 = cross_val_score(m, Xr, yr, cv=섞어나누기, scoring=____).mean()
    print(f'{이름:10s} R² {점수:.3f}')

## ④ 비지도 — 정답 없이 묶어 보면 종이 나뉠까? (k-평균 + PCA)
정답(종)을 **숨기고** 네 숫자만으로 세 무리를 만든 뒤, 실제 종과 얼마나 겹치는지 봅니다.
PCA는 네 숫자를 **두 축으로 줄여** 그림으로 볼 수 있게 합니다.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
Z = StandardScaler().fit_transform(X)
무리 = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(Z)
print(pd.crosstab(무리, y))          # 행: 만든 무리 / 열: 실제 종
P = PCA(2).fit_transform(Z)
plt.scatter(P[:, 0], P[:, 1], c=무리, cmap='viridis', s=15)
plt.title('k-means clusters on 2 PCA axes'); plt.show()

**생각해 보기 ②** 정답을 한 번도 안 보고 만든 무리가 실제 종과 얼마나 맞나요? 잘 안 나뉜 두 종은 무엇이고, 왜 그럴까요?

---
### 정답 예시 (막혔을 때만)
- ② `cv=5` · ③ `scoring='r2'`
- ③ 에서 `cv=5`(섞지 않음)로 바꿔 보면 R²가 크게 떨어집니다 — **데이터를 나누는 방법**도 결과를 바꿉니다.
- 펭귄 데이터는 쉬운 편이라 대부분 0.95 이상입니다. 이럴 땐 **가장 단순하고 설명하기 쉬운 모델**(로지스틱 회귀 · 트리)을 고르는 것이 «다섯 걸음»의 결론입니다.
- k-평균은 Gentoo는 거의 정확히 묶지만, **Adelie와 Chinstrap** 은 몸 크기가 비슷해 섞이기 쉽습니다.